Author: **[Write your name]**

# **Supervised Learning**
## **Building and Evaluating a Predictive Model on a New Dataset**

In this notebook, we will work with a **new dataset stored in Excel** and apply **Supervised Learning**.

The main objective is to build a model that learns from labeled examples and predicts a **target variable**.

This notebook is designed to work with either:

- **Classification** problems, where the target is categorical.
- **Regression** problems, where the target is numerical.

The workflow includes data loading, cleaning, preprocessing, train/test splitting, model training, evaluation, interpretation, and export of predictions.


# **1. What is Supervised Learning?**

**Supervised Learning** is a type of Machine Learning in which the model learns from examples that already contain the correct answer.

Each observation contains:

- **Features ($X$):** the input variables used to make a prediction.
- **Target ($y$):** the variable that we want to predict.

The model learns a relationship of the form:

$$
X \longrightarrow y
$$

For example:

| Age | Income | Previous Purchases | Will Buy |
|---:|---:|---:|---|
| 20 | 500 | 2 | No |
| 35 | 1500 | 8 | Yes |
| 50 | 3000 | 15 | Yes |

Here:

- `Age`, `Income`, and `Previous Purchases` are **features**.
- `Will Buy` is the **target**.

The algorithm learns from known examples and then tries to predict the target for new observations.


## **1.1 Classification vs Regression**

There are two main types of supervised learning problems.

### **Classification**

The target is a **category or class**.

Examples:

- Spam / Not Spam
- Survived / Did Not Survive
- Fraud / Not Fraud
- Disease type
- Customer category

### **Regression**

The target is a **numerical value**.

Examples:

- House price
- Temperature
- Salary
- Sales
- Population

This notebook automatically decides whether the problem looks like classification or regression after you define the target column.


# **2. Workflow**

1. Install and import libraries.
2. Load the Excel dataset.
3. Inspect available Excel sheets.
4. Explore the dataset.
5. Clean the data.
6. Define the target variable.
7. Separate features and target.
8. Identify numerical and categorical variables.
9. Split the data into training and testing sets.
10. Preprocess the features.
11. Train supervised learning models.
12. Compare model performance.
13. Evaluate the best model.
14. Interpret the model.
15. Save predictions.


## **3. Install Required Packages**

Run this cell only if the packages are not already installed.


In [ ]:
%pip install pandas numpy matplotlib scikit-learn openpyxl

## **4. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)
from sklearn.inspection import permutation_importance

print("Libraries imported successfully.")


# **5. Load the Dataset**

The main example uses an Excel file.

Change the path below so that it points to your own dataset.


In [ ]:
# ============================================================
# EXCEL FILE
# ============================================================

file_path = r"C:\Documentos\2SEM2026\intelligence Artificial\lab assigment 1\your_dataset.xlsx"


# ============================================================
# IF THE DATASET WERE A TXT FILE
# ============================================================

# TXT separated by tabs:
# data = pd.read_csv(
#     r"C:\Documentos\dataset.txt",
#     sep="\t"
# )

# TXT separated by commas:
# data = pd.read_csv(
#     r"C:\Documentos\dataset.txt",
#     sep=","
# )


## **5.1 Check the Available Excel Sheets**

Some Excel files contain multiple sheets. This cell displays all of them.


In [ ]:
excel_file = pd.ExcelFile(file_path)

print("Available sheets:")

for i, sheet in enumerate(excel_file.sheet_names):
    print(i, "->", sheet)


## **5.2 Select the Sheet**

By default, the first sheet is selected.

If your data are stored in another sheet, replace:

```python
sheet_name = excel_file.sheet_names[0]
```

with something like:

```python
sheet_name = "Data"
```


In [ ]:
sheet_name = excel_file.sheet_names[0]

data = pd.read_excel(
    file_path,
    sheet_name=sheet_name
)

print("Selected sheet:", sheet_name)
print("File read successfully.")


# **6. Initial Dataset Exploration**

Before building a model, we should understand the structure of the dataset.


In [ ]:
print("Dataset dimensions:", data.shape)

print("\nFirst 10 rows:")
display(data.head(10))


In [ ]:
print("Dataset information:")
data.info()


In [ ]:
print("Column names:")

for column in data.columns:
    print("-", column)


In [ ]:
print("Missing values per column:")
display(
    data.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)


# **7. Clean the Dataset**

We remove:

- Completely empty rows.
- Completely empty columns.
- Columns whose names start with `Unnamed`.
- Extra spaces in column names.


In [ ]:
clean_data = data.copy()

clean_data = clean_data.dropna(
    axis=0,
    how="all"
)

clean_data = clean_data.dropna(
    axis=1,
    how="all"
)

clean_data.columns = [
    str(column).strip()
    for column in clean_data.columns
]

unnamed_columns = [
    column
    for column in clean_data.columns
    if column.lower().startswith("unnamed")
]

clean_data = clean_data.drop(
    columns=unnamed_columns,
    errors="ignore"
)

clean_data = clean_data.reset_index(drop=True)

print("Dimensions after cleaning:", clean_data.shape)
display(clean_data.head())


## **7.1 Try to Convert Numerical Values Stored as Text**

Sometimes Excel imports numerical columns as text. The following block tries to convert a column when at least 80% of its valid values can be interpreted as numbers.


In [ ]:
for column in clean_data.columns:

    if clean_data[column].dtype == "object":

        converted = pd.to_numeric(
            clean_data[column],
            errors="coerce"
        )

        original_valid = clean_data[column].notna().sum()
        converted_valid = converted.notna().sum()

        if original_valid > 0:

            ratio = converted_valid / original_valid

            if ratio >= 0.80:
                clean_data[column] = converted

print("Conversion completed.")


# **8. Define the Target Variable**

This is the most important difference between supervised and unsupervised learning.

In supervised learning, we must specify the column that contains the answer we want to predict.

Replace:

```python
target_column = "PUT_TARGET_COLUMN_HERE"
```

with the exact name of your target column.

Examples:

```python
target_column = "Survived"
```

```python
target_column = "median_house_value"
```

```python
target_column = "Diagnosis"
```


In [ ]:
target_column = "PUT_TARGET_COLUMN_HERE"

if target_column not in clean_data.columns:

    print("Available columns:")

    for column in clean_data.columns:
        print("-", column)

    raise ValueError(
        "The target column was not found. "
        "Change target_column to one of the column names shown above."
    )

print("Target variable:", target_column)


# **9. Remove Rows with Missing Target Values**

A supervised learning model cannot train on observations whose correct target value is unknown.


In [ ]:
before = len(clean_data)

clean_data = clean_data.dropna(
    subset=[target_column]
).reset_index(drop=True)

after = len(clean_data)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed because the target was missing:", before - after)


# **10. Separate Features and Target**

We define:

$$
X = \text{input features}
$$

and

$$
y = \text{target}
$$


In [ ]:
X = clean_data.drop(
    columns=[target_column]
)

y = clean_data[target_column]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


# **11. Detect the Type of Supervised Learning Problem**

The notebook uses a practical rule:

- Non-numerical targets are treated as **classification**.
- Numerical targets with relatively few distinct values are also treated as **classification**.
- Numerical targets with many distinct values are treated as **regression**.

You can manually override the result if necessary.


In [ ]:
number_of_unique_targets = y.nunique(dropna=True)

if (
    y.dtype == "object"
    or str(y.dtype) == "category"
    or number_of_unique_targets <= 20
):
    problem_type = "classification"
else:
    problem_type = "regression"

print("Detected problem type:", problem_type)
print("Number of unique target values:", number_of_unique_targets)


### **11.1 Optional Manual Override**

If the automatic detection is not correct, uncomment one of these lines:

```python
# problem_type = "classification"
# problem_type = "regression"
```


In [ ]:
# Optional manual override:

# problem_type = "classification"
# problem_type = "regression"

print("Problem type used:", problem_type)


# **12. Remove Features That Are Not Useful**

Unique identifiers, names, codes, addresses, and record numbers often do not help prediction and may cause overfitting.

The following block automatically detects high-cardinality categorical columns.


In [ ]:
categorical_candidates = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

high_cardinality_columns = []

for column in categorical_candidates:

    unique_values = X[column].nunique(dropna=True)
    valid_values = X[column].notna().sum()

    if valid_values > 0:

        unique_ratio = unique_values / valid_values

        if unique_values > 30 and unique_ratio > 0.70:
            high_cardinality_columns.append(column)

print("Detected high-cardinality columns:")
print(high_cardinality_columns)


## **12.1 Columns to Exclude Manually**

You can add known identifiers or irrelevant variables here.

Example:

```python
columns_to_exclude_manually = ["ID", "Name"]
```


In [ ]:
columns_to_exclude_manually = []

columns_to_exclude = list(
    set(
        high_cardinality_columns
        + columns_to_exclude_manually
    )
)

X = X.drop(
    columns=columns_to_exclude,
    errors="ignore"
)

print("Excluded columns:")
print(columns_to_exclude)

print("\nRemaining feature columns:")
print(X.columns.tolist())


# **13. Identify Numerical and Categorical Features**

In [ ]:
numerical_columns = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

date_columns = X.select_dtypes(
    include=["datetime", "datetimetz"]
).columns.tolist()

print("Numerical features:")
print(numerical_columns)

print("\nCategorical features:")
print(categorical_columns)

print("\nDate features:")
print(date_columns)


## **13.1 Remove Date Columns from the Basic Model**

Dates usually need feature engineering before they can be used effectively.

For this introductory notebook, raw date columns are removed.


In [ ]:
X = X.drop(
    columns=date_columns,
    errors="ignore"
)

numerical_columns = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Final numerical features:")
print(numerical_columns)

print("\nFinal categorical features:")
print(categorical_columns)


# **14. Split the Dataset into Training and Testing Sets**

We do not train and evaluate the model on exactly the same data.

Instead, we separate the dataset into:

- **Training set:** used to teach the model.
- **Testing set:** used to evaluate how well the model generalizes to unseen data.

We will use:

$$
80\% \text{ training}
$$

and

$$
20\% \text{ testing}
$$


In [ ]:
if problem_type == "classification":

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=42,
            stratify=y
        )

    except ValueError:
        print(
            "Stratified split was not possible. "
            "Using a regular random split instead."
        )

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=42
        )

else:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))


# **15. Preprocessing**

### Numerical variables

- Missing values are replaced with the median.
- `StandardScaler` standardizes the variables.

### Categorical variables

- Missing values are replaced with the most frequent category.
- `OneHotEncoder` converts categories into numerical indicator variables.


In [ ]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

transformers = []

if len(numerical_columns) > 0:
    transformers.append(
        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        )
    )

if len(categorical_columns) > 0:
    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    )

if len(transformers) == 0:
    raise ValueError(
        "No usable features were found."
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

print("Preprocessor created successfully.")


# **16. Train Supervised Learning Models**

We will compare two models.

### If the problem is Classification

- Logistic Regression
- Random Forest Classifier

### If the problem is Regression

- Linear Regression
- Random Forest Regressor


In [ ]:
if problem_type == "classification":

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=2000
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    }

else:

    models = {
        "Linear Regression": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            random_state=42
        )
    }

print("Models:")
for name in models:
    print("-", name)


# **17. Train and Compare the Models**

Each model is placed inside a pipeline:

$$
\text{Raw Data}
\rightarrow
\text{Preprocessing}
\rightarrow
\text{Model}
$$


In [ ]:
trained_models = {}
model_scores = {}

for name, estimator in models.items():

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                estimator
            )
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    trained_models[name] = pipeline

    if problem_type == "classification":

        score = accuracy_score(
            y_test,
            predictions
        )

        model_scores[name] = score

        print(
            name,
            "-> Accuracy:",
            round(score, 4)
        )

    else:

        score = r2_score(
            y_test,
            predictions
        )

        model_scores[name] = score

        print(
            name,
            "-> R²:",
            round(score, 4)
        )


# **18. Select the Best Model**

For classification, the notebook selects the model with the highest **accuracy**.

For regression, it selects the model with the highest **$R^2$**.


In [ ]:
best_model_name = max(
    model_scores,
    key=model_scores.get
)

best_model = trained_models[
    best_model_name
]

print("Best model:", best_model_name)
print("Best score:", round(model_scores[best_model_name], 4))


# **19. Evaluate the Best Model**

The evaluation depends on whether the task is classification or regression.


In [ ]:
y_pred = best_model.predict(
    X_test
)

if problem_type == "classification":

    print("Accuracy:")
    print(
        round(
            accuracy_score(y_test, y_pred),
            4
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )

else:

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    mse = mean_squared_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        y_pred
    )

    print("Mean Absolute Error (MAE):", round(mae, 4))
    print("Mean Squared Error (MSE):", round(mse, 4))
    print("Root Mean Squared Error (RMSE):", round(rmse, 4))
    print("R²:", round(r2, 4))


# **20. Classification Visualization**

If the problem is classification, we display the **confusion matrix**.

The diagonal represents correct predictions.


In [ ]:
if problem_type == "classification":

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    fig, ax = plt.subplots(figsize=(6, 5))

    image = ax.imshow(cm)

    ax.set_title("Confusion Matrix")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    classes = np.unique(
        np.concatenate(
            [
                np.asarray(y_test),
                np.asarray(y_pred)
            ]
        )
    )

    ax.set_xticks(
        np.arange(len(classes))
    )

    ax.set_yticks(
        np.arange(len(classes))
    )

    ax.set_xticklabels(
        classes,
        rotation=45,
        ha="right"
    )

    ax.set_yticklabels(
        classes
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center"
            )

    plt.colorbar(image)
    plt.tight_layout()
    plt.show()

else:

    print(
        "This is a regression problem, so a confusion matrix is not used."
    )


# **21. Regression Visualization**

For regression, we compare the real target values with the predicted values.

A good model should produce points close to the diagonal line:

$$
y_{predicted} = y_{actual}
$$


In [ ]:
if problem_type == "regression":

    plt.figure(figsize=(7, 6))

    plt.scatter(
        y_test,
        y_pred
    )

    minimum = min(
        np.min(y_test),
        np.min(y_pred)
    )

    maximum = max(
        np.max(y_test),
        np.max(y_pred)
    )

    plt.plot(
        [minimum, maximum],
        [minimum, maximum]
    )

    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title("Actual vs Predicted Values")
    plt.grid()

    plt.show()

else:

    print(
        "This is a classification problem, so this regression graph is not used."
    )


# **22. Model Interpretation with Permutation Importance**

Permutation importance measures how much the model performance decreases when the values of one feature are randomly shuffled.

A larger importance means the model depends more strongly on that feature.

This approach works even when the model contains preprocessing steps.


In [ ]:
try:

    importance_result = permutation_importance(
        best_model,
        X_test,
        y_test,
        n_repeats=5,
        random_state=42
    )

    importance_table = pd.DataFrame(
        {
            "Feature": X_test.columns,
            "Importance": importance_result.importances_mean
        }
    )

    importance_table = importance_table.sort_values(
        "Importance",
        ascending=False
    )

    display(
        importance_table.head(15)
    )

except Exception as error:

    print(
        "Permutation importance could not be calculated:"
    )

    print(error)


## **22.1 Plot the Most Important Features**

In [ ]:
if "importance_table" in globals():

    top_features = importance_table.head(10)

    plt.figure(figsize=(8, 5))

    plt.barh(
        top_features["Feature"][::-1],
        top_features["Importance"][::-1]
    )

    plt.xlabel("Permutation Importance")
    plt.ylabel("Feature")
    plt.title("Most Important Features")
    plt.tight_layout()

    plt.show()


# **23. Create a Table with Predictions**

We add the real and predicted target values to the test observations.


In [ ]:
results = X_test.copy()

results[
    "Actual_" + target_column
] = y_test.values

results[
    "Predicted_" + target_column
] = y_pred

display(
    results.head(20)
)


# **24. Save the Results to Excel**

The output file contains:

- The test observations.
- The real target values.
- The model predictions.


In [ ]:
output_file = r"C:\Documentos\2SEM2026\intelligence Artificial\lab assigment 1\supervised_learning_predictions.xlsx"

results.to_excel(
    output_file,
    index=False
)

print("Results saved successfully at:")
print(output_file)


# ============================================================
# IF YOU WANT TO SAVE THE RESULTS AS TXT
# ============================================================

# results.to_csv(
#     r"C:\Documentos\supervised_learning_predictions.txt",
#     sep="\t",
#     index=False
# )


# **25. Optional: Predict New Data**

To predict a new observation, create a DataFrame using the same feature columns as the training dataset.

Example:

```python
new_observation = pd.DataFrame({
    "Age": [30],
    "Income": [2500],
    "Gender": ["Female"]
})

prediction = best_model.predict(new_observation)

print(prediction)
```

The exact column names and values depend on your dataset.


In [ ]:
# Example template only.
# Replace the column names and values with those from your dataset.

# new_observation = pd.DataFrame({
#     "Feature_1": [value_1],
#     "Feature_2": [value_2],
#     "Feature_3": [value_3]
# })

# prediction = best_model.predict(
#     new_observation
# )

# print(
#     "Prediction:",
#     prediction
# )


# **26. Analysis Questions**

### **1. What is the target variable in this dataset?**

**Answer:**  
...

---

### **2. Is this a classification or regression problem? Why?**

**Answer:**  
...

---

### **3. Which model obtained the best performance?**

**Answer:**  
...

---

### **4. Which evaluation metric did you use and what does it mean?**

For classification, you can discuss:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

For regression, you can discuss:

- MAE
- MSE
- RMSE
- $R^2$

**Answer:**  
...

---

### **5. Which features appear to be the most important?**

Use the permutation importance table.

**Answer:**  
...

---

### **6. Why do we divide the dataset into training and testing sets?**

**Answer:**  
...

---

### **7. What is the main difference between Supervised Learning and Unsupervised Learning?**

**Answer:**  
...

---

### **8. Conclusion**

Summarize:

- The problem type.
- The target variable.
- The best model.
- The main evaluation result.
- The most important observations from the analysis.

**Answer:**  
...


# **Procedure Summary**

The complete supervised learning workflow is:

$$
\text{Dataset}
\rightarrow
\text{Cleaning}
\rightarrow
\text{Features } X \text{ and Target } y
\rightarrow
\text{Train/Test Split}
\rightarrow
\text{Preprocessing}
\rightarrow
\text{Model Training}
\rightarrow
\text{Evaluation}
\rightarrow
\text{Prediction}
$$

The fundamental idea is that the model learns from observations for which the correct target is already known and then uses that learned relationship to make predictions on unseen data.
